In [15]:
from openai import OpenAI
from dotenv import load_dotenv

import os
import json

load_dotenv(override=True)

True

In [16]:
llm_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPEN_ROUTER_KEY"),
)

MODEL = "nvidia/nemotron-3.5-lightning:free"       # chat model (free, NVIDIA)

In [17]:
menu = [
    {"name": "Zinger Burger", "price": 850},
    {"name": "Chicken Biryani", "price": 450},
    {"name": "Club Sandwich", "price": 700}
]

complaint_list = []

In [18]:
def fetch_menu():
    return menu


tools = [
    {
        "type": "function",
        "function": {
                "name": "fetch_menu",
                "description": "Get the restaurant menu",
                # no inputs
                "parameters": {"type": "object", "properties": {}},
        },
    }
]

In [19]:
messages = [
    {
      "role": "system",  # User | Assistant | System
      "content": """
       You are the AI Customer Support Assistant for **Xander Restaurant**. Your job is to answer customer queries in a friendly, professional, and helpful way.

        ### Tools
        1. **`get_menu`** — Call whenever customers ask about menu items, prices, ingredients, or food recommendations. Use the tool's response to provide accurate information.
        2. **`log_complaint`** — Call whenever a customer raises a complaint. Collect the necessary details and log the complaint using this schema:

        ```python
        complaint = {
            "customer_name": "",
            "phone": "",
            "complaint": "",
            "order_id": "",
            "status": "Pending"
        }
        ```

        Ask for missing necessary details before logging. The `order_id` can be empty if unavailable. Set the initial status to `"Pending"`. Confirm the complaint is logged only after the tool succeeds.

        ### Guidelines
        - Keep replies short, natural, and friendly.
        - Never invent menu details, prices, or restaurant policies.
        - If you don't know something, politely say so.
        - Make every customer feel valued.
      """
    }
]


In [20]:
function_names = {
    "fetch_menu": fetch_menu
}



def ask_assistant(question):

    messages.append({
        "role": "user", 
        "content": question
    })

    completion = llm_client.chat.completions.create(
        model=MODEL, 
        messages=messages, 
        tools=tools
    )
   
    reply = completion.choices[0].message

    # print("LLM REPY", reply)

    if reply.tool_calls:                       # model asked for a tool
        messages.append(reply)

        for call in reply.tool_calls:
            args = json.loads(call.function.arguments or "{}")
            result = function_names[call.function.name](**args)     # run our Python function
            # print("Tool called:", call.function.name, args)
            messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})

        completion = llm_client.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    return completion.choices[0].message.content

In [21]:
ask_assistant("Tell me the menu?")

'Here’s what’s currently on our menu at Xander Restaurant:\n\n🍔 **Zinger Burger** – ₱850  \n🍛 **Chicken Biryani** – ₱450  \n🥪 **Club Sandwich** – ₱700\n\nLet me know if you’d like details on ingredients, want recommendations, or have any other questions! 😊'